In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from numpy.linalg import norm
from Image_transformation_git import crop_image_from_gray, circle_crop, get_brightness
from Image_transformation_git import adjust_brightness, hist_equalization_green_channel




In [ ]:
# -------------------------------
# ⚙️ Parameters
# -------------------------------
INPUT_DIR = '/content/drive/MyDrive/ROP-Data/images_stack/split_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/ROP-Data/split_dataset_preprocessed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGE_SIZE = (224, 224)  # Final resize shape
MIN_BRIGHTNESS = 120



In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from Image_transformation_git import crop_image_from_gray, circle_crop, get_brightness
from Image_transformation_git import adjust_brightness, hist_equalization_green_channel


# ✅ توابع wrap شده برای استفاده بدون matplotlib و فایل
def apply_circle_crop(img):
    img = crop_image_from_gray(img)
    img = circle_crop(img, sigmaX=30)
    return img

def apply_adjust_brightness(img):
    brightness = get_brightness(img)
    if brightness >= MIN_BRIGHTNESS:
        return img
    else:
        return cv2.convertScaleAbs(img, alpha=MIN_BRIGHTNESS / brightness,
                                   beta=(MIN_BRIGHTNESS - brightness))

def apply_clahe_green(img):
    r, g, b = cv2.split(img)
    zero_ch = np.zeros(img.shape[0:2], dtype="uint8")
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    g_clahe = clahe.apply(g)
    return cv2.merge([zero_ch, g_clahe, zero_ch])

# 🔧 pipeline نهایی
def preprocess_pipeline(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        return None

    try:
        img = apply_circle_crop(img)
        img = apply_adjust_brightness(img)
        img = apply_clahe_green(img)
        img = cv2.resize(img, IMAGE_SIZE)
        return img
    except Exception as e:
        print(f"❌ Error processing {img_path}: {e}")
        return None

# 🔄 پردازش کل داده‌ها
def preprocess_all_images(input_dir, output_dir):
    for class_name in sorted(os.listdir(input_dir)):
        class_input_path = os.path.join(input_dir, class_name)
        if not os.path.isdir(class_input_path):
            continue

        class_output_path = os.path.join(output_dir, class_name)
        os.makedirs(class_output_path, exist_ok=True)

        for fname in tqdm(os.listdir(class_input_path), desc=f"Processing [{class_name}]"):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in IMAGE_EXTS:
                continue

            input_path = os.path.join(class_input_path, fname)
            output_path = os.path.join(class_output_path, fname)

            if os.path.exists(output_path):
                continue  # تصویر قبلاً پردازش شده

            img = preprocess_pipeline(input_path)
            if img is not None:
                cv2.imwrite(output_path, img)



In [ ]:

# 🚀 اجرا
if __name__ == "__main__":
    preprocess_all_images(INPUT_DIR, OUTPUT_DIR)

Processing [Positive]: 100%|██████████| 60/60 [00:18<00:00,  3.23it/s]


In [ ]:
import os

# مسیر پوشه‌ها رو مشخص کن
processed_root = '/content/drive/MyDrive/ROP-Data/10_split_dataset_color'  # مسیر کلی
positive_dir = os.path.join(processed_root, 'Positive')
negative_dir = os.path.join(processed_root, 'Negative')

# پسوندهای معتبر تصاویر
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images_in_folder(folder_path):
    if not os.path.exists(folder_path):
        print(f"📁 مسیر وجود ندارد: {folder_path}")
        return 0
    return sum(1 for f in os.listdir(folder_path)
               if os.path.isfile(os.path.join(folder_path, f)) and os.path.splitext(f)[1].lower() in valid_exts)

# شمارش
num_positive = count_images_in_folder(positive_dir)
num_negative = count_images_in_folder(negative_dir)

# نمایش
print(f"✅ تعداد تصاویر پوشه مثبت (positive): {num_positive}")
print(f"❌ تعداد تصاویر پوشه منفی (negative): {num_negative}")


📁 مسیر وجود ندارد: /content/drive/MyDrive/ROP-Data/10_split_dataset_color/Positive
📁 مسیر وجود ندارد: /content/drive/MyDrive/ROP-Data/10_split_dataset_color/Negative
✅ تعداد تصاویر پوشه مثبت (positive): 0
❌ تعداد تصاویر پوشه منفی (negative): 0


In [ ]:
import os

# مسیر پوشه‌ها رو مشخص کن
processed_root = '/content/drive/MyDrive/ROP-Data/10_split_dataset_preprocessed_color'  # مسیر کلی
positive_dir = os.path.join(processed_root, 'Positive')
negative_dir = os.path.join(processed_root, 'Negative')

# پسوندهای معتبر تصاویر
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images_in_folder(folder_path):
    if not os.path.exists(folder_path):
        print(f"📁 مسیر وجود ندارد: {folder_path}")
        return 0
    return sum(1 for f in os.listdir(folder_path)
               if os.path.isfile(os.path.join(folder_path, f)) and os.path.splitext(f)[1].lower() in valid_exts)

# شمارش
num_positive = count_images_in_folder(positive_dir)
num_negative = count_images_in_folder(negative_dir)

# نمایش
print(f"✅ تعداد تصاویر پوشه مثبت (positive): {num_positive}")
print(f"❌ تعداد تصاویر پوشه منفی (negative): {num_negative}")


✅ تعداد تصاویر پوشه مثبت (positive): 563
❌ تعداد تصاویر پوشه منفی (negative): 1249
